In [0]:
#Using Snowflake with Kafka and Spark:

https://docs.snowflake.com/en/user-guide/connectors

------------------------------------------------------------------------------------------
https://docs.snowflake.com/en/user-guide/spark-connector

Snowflake Connector for Spark:  (ALTERNATIVE IS SNOWPARK API)
Bring Snowflake into the Apache Spark ecosystem, enabling Spark to read data from, and write data to, Snowflake.

--From Spark’s perspective, Snowflake looks similar to other Spark data sources (PostgreSQL, HDFS, S3, etc.).


Note:
As an alternative to using Spark, consider writing your code to use Snowpark API instead. Snowpark allows you to perform all of your work within Snowflake (rather than in a separate Spark compute cluster). Snowpark also supports pushdown of all operations, including Snowflake UDFs.

Snowflake supports three versions of Spark: Spark 3.2, Spark 3.3, and Spark 3.4. There is a separate version of the Snowflake connector for each version of Spark. Use the correct version of the connector for your version of Spark.

The connector runs as a Spark plugin and is provided as a Spark package (spark-snowflake).

Overview of the Spark Connector:
--The Snowflake Connector for Spark enables using Snowflake as an Apache Spark data source, similar to other data sources (PostgreSQL, HDFS, S3, etc.).

Interaction Between Snowflake and Spark:
The connector supports bi-directional data movement between a Snowflake cluster and a Spark cluster. The Spark cluster can be self-hosted or accessed through another service, such as Qubole, AWS EMR, or Databricks.

Using the connector, you can perform the following operations:
Populate a Spark DataFrame from a table (or query) in Snowflake.
Write the contents of a Spark DataFrame to a table in Snowflake.

The connector uses Scala 2.12.x or 2.13.x to perform these operations and uses the Snowflake JDBC driver to communicate with Snowflake.

Note

The Snowflake Connector for Spark is not strictly required to connect Snowflake and Apache Spark; other 3rd-party JDBC drivers can be used. However, we recommend using the Snowflake Connector for Spark because the connector, in conjunction with the Snowflake JDBC driver, has been optimized for transferring large amounts of data between the two systems. It also provides enhanced performance by supporting query pushdown from Spark into Snowflake.

Data Transfer:
The Snowflake Spark Connector supports two transfer modes:

Internal transfer uses a temporary location created and managed internally/transparently by Snowflake.
External transfer uses a storage location, usually temporary, created and managed by the user.


Internal Data Transfer:
The transfer of data between the two systems is facilitated through a Snowflake internal stage that the connector automatically creates and manages:
--Upon connecting to Snowflake and initializing a session in Snowflake, the connector creates the internal stage.
--Throughout the duration of the Snowflake session, the connector uses the stage to store data while transferring it to its destination.
--At the end of the Snowflake session, the connector drops the stage, thereby removing all the temporary data in the stage.

Note that support for internal transfer requires a specific version (or higher) of the connector, based on the cloud platform for your Snowflake account:

AWS
The internal data transfer mode is supported only in version 2.2.0 (and higher) of the connector.

External Data Transfer:
The transfer of data between the two systems is facilitated through a storage location that the user specifies and files automatically created by the connector:

AWS:
Transfer data files are created and stored in an S3 bucket.


Column Mapping:
--When you copy data from a Spark table to a Snowflake table, if the column names do not match, you can map column names from Spark to Snowflake using the columnmapping parameter
--Column mapping is supported only for internal data transfer.

Query Pushdown:
For optimal performance, you typically want to avoid reading lots of data or transferring large intermediate results between systems. Ideally, most of the processing should happen close to where the data is stored to leverage the capabilities of the participating stores to dynamically eliminate data that is not needed.

Query pushdown leverages these performance efficiencies by enabling large and complex Spark logical plans (in their entirety or in parts) to be processed in Snowflake, thus using Snowflake to do most of the actual work.

Query pushdown is supported in Version 2.1.0 (and higher) of the Snowflake Connector for Spark.

Pushdown is not possible in all situations. For example, Spark UDFs cannot be pushed down to Snowflake.

Note:
If you need pushdown for all operations, consider writing your code to use Snowpark API instead. Snowpark also supports pushdown of Snowflake UDFs.

Databricks Integration:
Databricks has integrated the Snowflake Connector for Spark into the Databricks Unified Analytics Platform to provide native connectivity between Spark and Snowflake.



Requirements:
--Snowflake Connector for Spark.
--Snowflake JDBC Driver (the version compatible with the version of the connector).
--Apache Spark environment, either self-hosted or hosted in any of the following:
	Qubole Data Service.
	Databricks.
	Amazon EMR.
--you can use a dedicated Amazon S3 bucket or Azure Blob storage container as a staging zone between the two systems; however, this is not required with version 2.2.0 (and higher) of the connector, which uses a temporary Snowflake internal stage (by default) for all data exchange.
--The role used in the connection needs USAGE and CREATE STAGE privileges on the schema that contains the table that you will read from or write to.

Note:
If you are using Databricks or Qubole to host Spark, you do not need to download or install the Snowflake Connector for Spark (or any of the other requirements). Both Databricks and Qubole have integrated the connector to provide native connectivity.


Configuring Snowflake for Spark in Databricks:
--The Databricks version 4.2 native Snowflake Connector allows your Databricks account to read data from and write data to Snowflake without importing any libraries.

The connector automatically distributes processing across Spark and Snowflake, without requiring the user to specify the parts of the processing that should be done on each system. Queries also benefit from Snowflake’s automatic query pushdown optimization.


Databricks documentation:
Read and write data from Snowflake:
https://docs.databricks.com/en/connect/external-systems/snowflake.html

Query a Snowflake table in Databricks:


# The following example applies to Databricks Runtime 11.3 LTS and above.

snowflake_table = (spark.read
  .format("snowflake")
  .option("host", "hostname")
  .option("port", "port") # Optional - will use default port 443 if not specified.
  .option("user", "username")
  .option("password", "password")
  .option("sfWarehouse", "warehouse_name")
  .option("database", "database_name")
  .option("schema", "schema_name") # Optional - will use default schema "public" if not specified.
  .option("dbtable", "table_name")
  .load()
)

# The following example applies to Databricks Runtime 10.4 and below.

snowflake_table = (spark.read
  .format("snowflake")
  .option("dbtable", table_name)
  .option("sfUrl", database_host_url)
  .option("sfUser", username)
  .option("sfPassword", password)
  .option("sfDatabase", database_name)
  .option("sfSchema", schema_name)
  .option("sfWarehouse", warehouse_name)
  .load()
)


------------------------------------------------------------------------------------------
Snowflake Connector for Kafka:
--Read data from one or more Apache Kafka topics and load the data into a Snowflake table.

Introduction to Apache Kafka:
--Apache Kafka software uses a publish and subscribe model to write and read streams of records, similar to a message queue or enterprise messaging system.
--Kafka allows processes to read and write messages asynchronously.
--A subscriber does not need to be connected directly to a publisher; a publisher can queue a message in Kafka for the subscriber to receive later.

--An application publishes messages to a topic, and an application subscribes to a topic to receive those messages. Kafka can process, as well as transmit, messages;

--Kafka Connect is a framework for connecting Kafka with external systems, including databases. A Kafka Connect cluster is a separate cluster from the Kafka cluster. The Kafka Connect cluster supports running and scaling out connectors (components that support reading and/or writing between external systems).

--The Kafka connector is designed to run in a Kafka Connect cluster to read data from Kafka topics and write the data into Snowflake tables.

Snowflake provides two versions of the connector:
--A version for the Confluent package version of Kafka.
--A version for the open source software (OSS) Apache Kafka package.

--From the perspective of Snowflake, a Kafka topic produces a stream of rows to be inserted into a Snowflake table. In general, each Kafka message contains one row.

--Kafka, like many message publish/subscribe platforms, allows a many-to-many relationship between publishers and subscribers. A single application can publish to many topics, and a single application can subscribe to multiple topics. With Snowflake, the typical pattern is that one topic supplies messages (rows) for one Snowflake table.

--The current version of the Kafka connector is limited to loading data into Snowflake. The Kafka connector supports two data loading methods:
	--Snowpipe
	--Snowpipe Streaming.

Target tables for Kafka topics:
--Kafka topics can be mapped to existing Snowflake tables in the Kafka configuration. If the topics are not mapped, then the Kafka connector creates a new table for each topic using the topic name.

Schema of tables for Kafka topics:
--When you ingest into an Iceberg table, the schema includes the same default columns (record_content and record_metadata). However, they are structured type columns instead of VARIANT.

By default, with Snowpipe or Snowpipe Streaming, every Snowflake table loaded by the Kafka connector has a schema consisting of two VARIANT columns:

RECORD_CONTENT. This contains the Kafka message.
RECORD_METADATA. This contains metadata about the message, for example, the topic from which the message was read.

Workflow for the Kafka connector:
The Kafka connector completes the following process to subscribe to Kafka topics and create Snowflake objects:
--The Kafka connector subscribes to one or more Kafka topics based on the configuration information provided via the Kafka configuration file or command.

The connector creates the following objects for each topic:
--One internal stage to temporarily store data files for each topic.
--One pipe to ingest the data files for each topic partition.
--One table for each topic. If the table specified for each topic does not exist, the connector creates it; otherwise, the connector creates the RECORD_CONTENT and RECORD_METADATA columns in the existing table and verifies that the other columns are nullable (and produces an error if they are not).

notes:
--One or more applications publish JSON or Avro records to a Kafka cluster. The records are split into one or more topic partitions.
--The Kafka connector buffers messages from the Kafka topics. When a threshold (time or memory or number of messages) is reached, the connector writes the messages to a temporary file in the internal stage. The connector triggers Snowpipe to ingest the temporary file. Snowpipe copies a pointer to the data file into a queue.
--A Snowflake-provided virtual warehouse loads data from the staged file into the target table (i.e. the table specified in the configuration file for the topic) via the pipe created for the Kafka topic partition.
--(Not shown) The connector monitors Snowpipe and deletes each file in the internal stage after confirming that the file data was loaded into the table.
--If a failure prevented the data from loading, the connector moves the file into the table stage and produces an error message.

Note:
Snowflake polls the insertReport API for one hour. If the status of an ingested file does not succeed within this hour, the files being ingested are moved to a table stage.

It may take at least one hour for these files to be available on the table stage. Files are only moved to the table stage when their ingestion status could not be found within the previous hour.

Fault tolerance:
Both Kafka and the Kafka connector are fault-tolerant. Messages are neither duplicated nor silently dropped.

Data deduplication logic in the Snowpipe workflow in the data loading chain eliminates duplicate copies of repeating data except in rare cases. If an error is detected while Snowpipe loads a record (for example, the record was not well-formed JSON or Avro), then the record is not loaded; instead, the record is moved to a table stage.

The Kafka connector with Snowpipe Streaming supports dead-letter queues (DLQ) for error handling. For more information, refer to Error Handling and DLQ Properties for the Kafka Connector with Snowpipe Streaming.

Limitations of fault tolerance with the connector:
--Kafka Topics can be configured with a limit on storage space or retention time.
--The default retention time is 7 days. If the system is offline for more than the retention time, then expired records will not be loaded. Similarly, if Kafka’s storage space limit is exceeded, some messages will not be delivered.
--If messages in the Kafka topic are deleted or updated, these changes might not be reflected in the Snowflake table.

Note:
Instances of the Kafka connector do not communicate with each other. If you start multiple instances of the connector on the same topics or partitions, then multiple copies of the same row might be inserted into the table. This is not recommended; each topic should be processed by only one instance of the connector.

Troubleshooting the Kafka connector:
https://docs.snowflake.com/en/user-guide/kafka-connector-ts










DLQ:
https://docs.snowflake.com/en/user-guide/data-load-snowpipe-streaming-kafka#label-snowpipe-streaming-kafka-dlq-properties:























Spark Notes:

Spark SQL:
--Industries are using Hadoop extensively to analyze their data sets. The reason is that Hadoop framework is based on a simple programming model (MapReduce) and it enables a computing solution that is scalable, flexible, fault-tolerant and cost effective.
--Spark was introduced by Apache Software Foundation for speeding up the Hadoop computational computing software process.
--Spark is not a modified version of Hadoop and is not, really, dependent on Hadoop because it has its own cluster management. Hadoop is just one of the ways to implement Spark.
--Spark uses Hadoop in two ways – one is storage and second is processing. Since Spark has its own cluster management computation, it uses Hadoop for storage purpose only.

Apache Spark
Apache Spark is a lightning-fast cluster computing technology, designed for fast computation. It is based on Hadoop MapReduce and it extends the MapReduce model to efficiently use it for more types of computations, which includes interactive queries and stream processing. The main feature of Spark is its in-memory cluster computing that increases the processing speed of an application.

Features of Apache Spark:
Speed − Spark helps to run an application in Hadoop cluster, up to 100 times faster in memory, and 10 times faster when running on disk. This is possible by reducing number of read/write operations to disk. It stores the intermediate processing data in memory.

Supports multiple languages − Spark provides built-in APIs in Java, Scala, or Python. Therefore, you can write applications in different languages. Spark comes up with 80 high-level operators for interactive querying.

Advanced Analytics − Spark not only supports ‘Map’ and ‘reduce’. It also supports SQL queries, Streaming data, Machine learning (ML), and Graph algorithms.

Spark Built on Hadoop:
Diagram

There are three ways of Spark deployment as explained below.
Standalone − Spark Standalone deployment means Spark occupies the place on top of HDFS(Hadoop Distributed File System) and space is allocated for HDFS, explicitly. Here, Spark and MapReduce will run side by side to cover all spark jobs on cluster.

Hadoop Yarn − Hadoop Yarn deployment means, simply, spark runs on Yarn without any pre-installation or root access required. It helps to integrate Spark into Hadoop ecosystem or Hadoop stack. It allows other components to run on top of stack.

Spark in MapReduce (SIMR) − Spark in MapReduce is used to launch spark job in addition to standalone deployment. With SIMR, user can start Spark and uses its shell without any administrative access.

Spark SQL - Introduction:
Spark introduces a programming module for structured data processing called Spark SQL. It provides a programming abstraction called DataFrame and can act as distributed SQL query engine.

Features of Spark SQL:
Integrated − Seamlessly mix SQL queries with Spark programs. Spark SQL lets you query structured data as a distributed dataset (RDD) in Spark, with integrated APIs in Python, Scala and Java. This tight integration makes it easy to run SQL queries alongside complex analytic algorithms.

Unified Data Access − Load and query data from a variety of sources. Schema-RDDs provide a single interface for efficiently working with structured data, including Apache Hive tables, parquet files and JSON files.

Hive Compatibility − Run unmodified Hive queries on existing warehouses. Spark SQL reuses the Hive frontend and MetaStore, giving you full compatibility with existing Hive data, queries, and UDFs. Simply install it alongside Hive.

Standard Connectivity − Connect through JDBC or ODBC. Spark SQL includes a server mode with industry standard JDBC and ODBC connectivity.

Scalability − Use the same engine for both interactive and long queries. Spark SQL takes advantage of the RDD model to support mid-query fault tolerance, letting it scale to large jobs too. Do not worry about using a different engine for historical data.

SPARK SQL ARCHITECTURE:

https://www.tutorialspoint.com/spark_sql/spark_sql_introduction.htm




















----------------------------------------------------------------------------------------
Snowflake Connector for Kafka:
Read data from one or more Apache Kafka topics and load the data into a Snowflake table.

----------------------------------------------------------------------------------------
Spark Interview Questions:

Q. while ingesting customer data from an external source, if any duplicate entries.
need to remove duplicates and retain only latest entry based on a timestamp column?

data = [("101","2023-12-01",100),("101","2023-12-02",150),("102","2023-12-01",200),("102","2023-12-02",250)]
columns = ["product_id","date","sales"]
df = spark.createDataFrame(data,columns)
df.display()


--------------==============SNOWPARK===========----------------------
Snowpark introduction:
--The Snowpark library provides an intuitive API for querying and processing data in in data pipeline.
--Using this library , can build applications that process data in snowflake without moving data to the system where your application code runs.
--Snowpark operations are executed lazily on the server, which reduces the amount of data transferred between client and snowflake database.
--Snowpark does not require a seperate cluster outside of snowflake for computations. all of the computations are done within snowflake.
--the core abstraction in snowpark is the dataframe, which represents a set of data and provides methods to operate on that data.


installation:
Python 3.8  --https://www.python.org/downloads/
Vs code: https://code.visualstudio.com/download
Install snowpark.
pip install snowflake-snowpark-python
or 
conda install snowflake-snowpark-python

pip install "snowflake-snowpark-python[pandas]"
or 
conda install snowflake-snowpark-python pandas pyarrow


